In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))   # so can be importted from src/

import numpy as np
import pandas as pd
from src.data_prep import load_and_clean, FEATURES, TARGET

In [2]:
df = load_and_clean()
X = df[FEATURES]
y = np.log1p(df[TARGET])          # latih pada skala log

print(X.shape, y.shape)

(1458, 5) (1458,)


In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("train:", X_train.shape, " test:", X_test.shape)

train: (1166, 5)  test: (292, 5)


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cat = ["Neighborhood"]
num = ["GrLivArea", "BedroomAbvGr", "Bathrooms", "GarageCars"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ("num", "passthrough", num),
])

results = {}

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = np.expm1(model.predict(X_test))     # balik ke dolar
    actual = np.expm1(y_test)
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    print(f"{name:20s} MAE=${mae:>9,.0f}  RMSE=${rmse:>9,.0f}  R2={r2:6.3f}")

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Dummy (mean)":     DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest":     RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

for name, m in models.items():
    evaluate(name, Pipeline([("pre", pre), ("model", m)]))

Dummy (mean)         MAE=$   57,012  RMSE=$   75,775  R2=-0.039
Linear Regression    MAE=$   24,769  RMSE=$   34,799  R2= 0.781
Random Forest        MAE=$   23,681  RMSE=$   33,337  R2= 0.799
Gradient Boosting    MAE=$   24,493  RMSE=$   33,630  R2= 0.795


In [6]:
pd.DataFrame(results).T.sort_values("R2", ascending=False).round(3)

,MAE,RMSE,R2
Random Forest,23681.466,33336.865,0.799
Gradient Boosting,24492.910,33630.068,0.795
Linear Regression,24768.519,34798.542,0.781
Dummy (mean),57012.205,75774.939,-0.039
